# WoWAH Retention Cohort Analysis

Bu notebook, full `fct_player_daily` katmanindan cohort retention tablolarini uretir ve final Parquet/CSV ciktilarini yazar.


## Cohort Mantigi

- **Cohort**: Ayni `first_seen_date` uzerinden ilk kez gorulen avatar grubudur.
- **D1 / D7 / D14 / D30 retention**: Cohort icindeki avatarlarin tam olarak 1, 7, 14 ve 30 gun sonra tekrar aktif olma oranidir.
- **Burn-in period**: Datasetin ilk 30 gunundeki cohort'lar final ozetten cikartilir; cunku dataset baslangicindan once aktif olan avatarlar yanlislikla yeni oyuncu gibi gorunebilir.
- **Immature cohorts**: Dataset sonuna yakin cohort'larda D30 gibi ufuklar henuz gozlemlenemedigi icin bu retention alanlari `NULL` birakilir.
- **Player proxy**: Kararli oyuncu kimligi icin `avatar_id` kullanilir; retention hesaplari bunun uzerinden yapilir.


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

print('DuckDB ve pandas hazir.')


In [ ]:
project_root = Path.cwd()
processed_dir = project_root / 'data/processed'
outputs_dir = project_root / 'data/outputs'
sql_file = project_root / 'sql/03_cohort_retention.sql'

fct_source_path = processed_dir / 'fct_player_daily.parquet'
if not fct_source_path.exists():
    raise FileNotFoundError(f'fct_player_daily parquet bulunamadi: {fct_source_path}')

cohort_output_path = processed_dir / 'agg_cohort_retention.parquet'
curve_output_path = processed_dir / 'agg_retention_curve.parquet'
validation_output_path = outputs_dir / 'retention_validation_summary.csv'

print('Input source:', fct_source_path)
print('SQL file:', sql_file)
print('Cohort output:', cohort_output_path)
print('Curve output:', curve_output_path)
print('Validation output:', validation_output_path)


In [ ]:
con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE VIEW fct_player_daily AS SELECT * FROM read_parquet('{fct_source_path}')"
)

con.execute(
    '''
    CREATE OR REPLACE VIEW retention_source_validation AS
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT avatar_id) AS unique_avatars,
        MIN(activity_date) AS min_activity_date,
        MAX(activity_date) AS max_activity_date,
        COUNT(*) - COUNT(DISTINCT (CAST(activity_date AS VARCHAR) || '|' || CAST(avatar_id AS VARCHAR))) AS duplicate_avatar_day_rows,
        SUM(CASE WHEN level_gain_day < 0 THEN 1 ELSE 0 END) AS negative_level_gain_day_count
    FROM fct_player_daily
    '''
)

source_validation_overview = con.execute(
    'SELECT * FROM retention_source_validation'
).df()

source_validation_overview


In [ ]:
con.execute(sql_file.read_text())

retention_model_overview = con.execute(
    '''
    SELECT
        (SELECT COUNT(*) FROM cohort_sizes) AS total_cohorts_before_filtering,
        (SELECT COUNT(*) FROM agg_cohort_retention) AS total_cohorts_after_burn_in,
        (SELECT COUNT(*) FROM agg_retention_curve) AS retention_curve_rows
    '''
).df()

print('Retention views SQL file uzerinden olusturuldu.')
retention_model_overview


In [ ]:
cohort_preview = con.execute(
    '''
    SELECT *
    FROM agg_cohort_retention
    ORDER BY cohort_date ASC
    LIMIT 10
    '''
).df()

curve_preview = con.execute(
    '''
    SELECT *
    FROM agg_retention_curve
    ORDER BY cohort_date ASC, cohort_age_day ASC
    LIMIT 10
    '''
).df()

print('Cohort retention preview')
print(cohort_preview.to_string(index=False))
print()
print('Retention curve preview')
print(curve_preview.to_string(index=False))


In [ ]:
con.execute(
    '''
    CREATE OR REPLACE VIEW retention_validation_overview AS
    SELECT
        (SELECT COUNT(*) FROM cohort_sizes) AS total_cohorts_before_filtering,
        (SELECT COUNT(*) FROM agg_cohort_retention) AS total_cohorts_after_burn_in_filtering,
        (SELECT MIN(cohort_date) FROM agg_cohort_retention) AS min_cohort_date,
        (SELECT MAX(cohort_date) FROM agg_cohort_retention) AS max_cohort_date,
        (SELECT AVG(cohort_size * 1.0) FROM agg_cohort_retention) AS average_cohort_size,
        (SELECT MEDIAN(cohort_size) FROM agg_cohort_retention) AS median_cohort_size,
        (SELECT AVG(d1_retention) FROM agg_cohort_retention) AS average_d1_retention,
        (SELECT AVG(d7_retention) FROM agg_cohort_retention) AS average_d7_retention,
        (SELECT AVG(d14_retention) FROM agg_cohort_retention) AS average_d14_retention,
        (SELECT AVG(d30_retention) FROM agg_cohort_retention) AS average_d30_retention,
        (SELECT SUM(CASE WHEN eligible_d1 THEN 1 ELSE 0 END) FROM cohort_retention_summary_model WHERE passes_burn_in) AS cohorts_eligible_for_d1,
        (SELECT SUM(CASE WHEN eligible_d7 THEN 1 ELSE 0 END) FROM cohort_retention_summary_model WHERE passes_burn_in) AS cohorts_eligible_for_d7,
        (SELECT SUM(CASE WHEN eligible_d14 THEN 1 ELSE 0 END) FROM cohort_retention_summary_model WHERE passes_burn_in) AS cohorts_eligible_for_d14,
        (SELECT SUM(CASE WHEN eligible_d30 THEN 1 ELSE 0 END) FROM cohort_retention_summary_model WHERE passes_burn_in) AS cohorts_eligible_for_d30
    '''
)

con.execute(
    '''
    CREATE OR REPLACE VIEW retention_sanity_checks AS
    SELECT
        (SELECT COUNT(*) FROM cohort_avatar_daily WHERE cohort_age_day < 0) AS negative_cohort_age_rows,
        (SELECT COUNT(*) FROM agg_retention_curve WHERE cohort_size <= 0) AS curve_nonpositive_cohort_size_rows,
        (SELECT COUNT(*) FROM agg_retention_curve WHERE retained_avatars > cohort_size) AS curve_retained_gt_cohort_size_rows,
        (SELECT COUNT(*) FROM agg_retention_curve WHERE retention_rate < 0 OR retention_rate > 1) AS curve_retention_rate_out_of_bounds_rows,
        (SELECT COUNT(*) FROM agg_cohort_retention WHERE cohort_size <= 0) AS summary_nonpositive_cohort_size_rows,
        (
            SELECT COUNT(*)
            FROM agg_cohort_retention
            WHERE (d1_retention IS NOT NULL AND (d1_retention < 0 OR d1_retention > 1))
               OR (d7_retention IS NOT NULL AND (d7_retention < 0 OR d7_retention > 1))
               OR (d14_retention IS NOT NULL AND (d14_retention < 0 OR d14_retention > 1))
               OR (d30_retention IS NOT NULL AND (d30_retention < 0 OR d30_retention > 1))
        ) AS summary_retention_rate_out_of_bounds_rows,
        (
            SELECT COUNT(*)
            FROM cohort_retention_summary_model
            WHERE passes_burn_in AND NOT eligible_d1 AND (retained_d1 IS NOT NULL OR d1_retention IS NOT NULL)
        ) AS immature_d1_populated_rows,
        (
            SELECT COUNT(*)
            FROM cohort_retention_summary_model
            WHERE passes_burn_in AND NOT eligible_d7 AND (retained_d7 IS NOT NULL OR d7_retention IS NOT NULL)
        ) AS immature_d7_populated_rows,
        (
            SELECT COUNT(*)
            FROM cohort_retention_summary_model
            WHERE passes_burn_in AND NOT eligible_d14 AND (retained_d14 IS NOT NULL OR d14_retention IS NOT NULL)
        ) AS immature_d14_populated_rows,
        (
            SELECT COUNT(*)
            FROM cohort_retention_summary_model
            WHERE passes_burn_in AND NOT eligible_d30 AND (retained_d30 IS NOT NULL OR d30_retention IS NOT NULL)
        ) AS immature_d30_populated_rows,
        (SELECT negative_level_gain_day_count FROM retention_source_validation) AS negative_input_level_gain_day_count
    '''
)

validation_overview = con.execute('SELECT * FROM retention_validation_overview').df()
sanity_checks = con.execute('SELECT * FROM retention_sanity_checks').df()

print('Validation overview')
print(validation_overview.to_string(index=False))
print()
print('Sanity checks')
print(sanity_checks.to_string(index=False))


In [ ]:
con.execute(
    '''
    CREATE OR REPLACE VIEW top_10_cohort_dates_by_size AS
    SELECT *
    FROM agg_cohort_retention
    ORDER BY cohort_size DESC, cohort_date ASC
    LIMIT 10
    '''
)

con.execute(
    '''
    CREATE OR REPLACE VIEW top_10_d7_retention_cohorts AS
    SELECT *
    FROM agg_cohort_retention
    WHERE cohort_size >= 100
      AND d7_retention IS NOT NULL
    ORDER BY d7_retention DESC, cohort_date ASC
    LIMIT 10
    '''
)

con.execute(
    '''
    CREATE OR REPLACE VIEW bottom_10_d7_retention_cohorts AS
    SELECT *
    FROM agg_cohort_retention
    WHERE cohort_size >= 100
      AND d7_retention IS NOT NULL
    ORDER BY d7_retention ASC, cohort_date ASC
    LIMIT 10
    '''
)

top_10_cohort_dates_by_size = con.execute('SELECT * FROM top_10_cohort_dates_by_size').df()
top_10_d7_retention_cohorts = con.execute('SELECT * FROM top_10_d7_retention_cohorts').df()
bottom_10_d7_retention_cohorts = con.execute('SELECT * FROM bottom_10_d7_retention_cohorts').df()

print('Top 10 cohort dates by cohort size')
print(top_10_cohort_dates_by_size.to_string(index=False))
print()
print('Top 10 cohort dates by D7 retention where cohort_size >= 100')
print(top_10_d7_retention_cohorts.to_string(index=False))
print()
print('Bottom 10 cohort dates by D7 retention where cohort_size >= 100')
print(bottom_10_d7_retention_cohorts.to_string(index=False))


In [ ]:
cohort_output_tmp = cohort_output_path.with_suffix(cohort_output_path.suffix + '.tmp')
curve_output_tmp = curve_output_path.with_suffix(curve_output_path.suffix + '.tmp')
validation_output_tmp = validation_output_path.with_suffix(validation_output_path.suffix + '.tmp')

for temp_path in [cohort_output_tmp, curve_output_tmp, validation_output_tmp]:
    if temp_path.exists():
        temp_path.unlink()

con.execute(
    f"COPY (SELECT * FROM agg_cohort_retention ORDER BY cohort_date) TO '{cohort_output_tmp}' (FORMAT PARQUET)"
)
con.execute(
    f"COPY (SELECT * FROM agg_retention_curve ORDER BY cohort_date, cohort_age_day) TO '{curve_output_tmp}' (FORMAT PARQUET)"
)

con.execute(
    '''
    CREATE OR REPLACE VIEW retention_validation_summary AS
    WITH summary_rows AS (
        SELECT
            'summary' AS section,
            1 AS position,
            'total_cohorts_before_filtering' AS metric_name,
            CAST(total_cohorts_before_filtering AS VARCHAR) AS metric_value,
            NULL::DATE AS cohort_date,
            NULL::BIGINT AS cohort_size,
            NULL::BIGINT AS retained_d1,
            NULL::BIGINT AS retained_d7,
            NULL::BIGINT AS retained_d14,
            NULL::BIGINT AS retained_d30,
            NULL::DOUBLE AS d1_retention,
            NULL::DOUBLE AS d7_retention,
            NULL::DOUBLE AS d14_retention,
            NULL::DOUBLE AS d30_retention,
            NULL::DOUBLE AS avg_level_gain_7d,
            NULL::DOUBLE AS avg_active_days_7d,
            NULL::DOUBLE AS avg_observations_7d
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 2, 'total_cohorts_after_burn_in_filtering', CAST(total_cohorts_after_burn_in_filtering AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 3, 'min_cohort_date', CAST(min_cohort_date AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 4, 'max_cohort_date', CAST(max_cohort_date AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 5, 'average_cohort_size', CAST(average_cohort_size AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 6, 'median_cohort_size', CAST(median_cohort_size AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 7, 'average_d1_retention', CAST(average_d1_retention AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 8, 'average_d7_retention', CAST(average_d7_retention AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 9, 'average_d14_retention', CAST(average_d14_retention AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 10, 'average_d30_retention', CAST(average_d30_retention AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 11, 'cohorts_eligible_for_d1', CAST(cohorts_eligible_for_d1 AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 12, 'cohorts_eligible_for_d7', CAST(cohorts_eligible_for_d7 AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 13, 'cohorts_eligible_for_d14', CAST(cohorts_eligible_for_d14 AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
        UNION ALL
        SELECT 'summary', 14, 'cohorts_eligible_for_d30', CAST(cohorts_eligible_for_d30 AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM retention_validation_overview
    ),
    top_cohort_size_rows AS (
        SELECT
            'top_10_cohort_dates_by_size' AS section,
            ROW_NUMBER() OVER (ORDER BY cohort_size DESC, cohort_date ASC) AS position,
            'cohort' AS metric_name,
            CAST(cohort_size AS VARCHAR) AS metric_value,
            cohort_date,
            cohort_size,
            retained_d1,
            retained_d7,
            retained_d14,
            retained_d30,
            d1_retention,
            d7_retention,
            d14_retention,
            d30_retention,
            avg_level_gain_7d,
            avg_active_days_7d,
            avg_observations_7d
        FROM top_10_cohort_dates_by_size
    ),
    top_d7_rows AS (
        SELECT
            'top_10_d7_retention_cohorts' AS section,
            ROW_NUMBER() OVER (ORDER BY d7_retention DESC, cohort_date ASC) AS position,
            'cohort' AS metric_name,
            CAST(d7_retention AS VARCHAR) AS metric_value,
            cohort_date,
            cohort_size,
            retained_d1,
            retained_d7,
            retained_d14,
            retained_d30,
            d1_retention,
            d7_retention,
            d14_retention,
            d30_retention,
            avg_level_gain_7d,
            avg_active_days_7d,
            avg_observations_7d
        FROM top_10_d7_retention_cohorts
    ),
    bottom_d7_rows AS (
        SELECT
            'bottom_10_d7_retention_cohorts' AS section,
            ROW_NUMBER() OVER (ORDER BY d7_retention ASC, cohort_date ASC) AS position,
            'cohort' AS metric_name,
            CAST(d7_retention AS VARCHAR) AS metric_value,
            cohort_date,
            cohort_size,
            retained_d1,
            retained_d7,
            retained_d14,
            retained_d30,
            d1_retention,
            d7_retention,
            d14_retention,
            d30_retention,
            avg_level_gain_7d,
            avg_active_days_7d,
            avg_observations_7d
        FROM bottom_10_d7_retention_cohorts
    )
    SELECT * FROM summary_rows
    UNION ALL
    SELECT * FROM top_cohort_size_rows
    UNION ALL
    SELECT * FROM top_d7_rows
    UNION ALL
    SELECT * FROM bottom_d7_rows
    '''
)

con.execute(
    f"""
    COPY (
        SELECT *
        FROM retention_validation_summary
        ORDER BY
            CASE section
                WHEN 'summary' THEN 1
                WHEN 'top_10_cohort_dates_by_size' THEN 2
                WHEN 'top_10_d7_retention_cohorts' THEN 3
                WHEN 'bottom_10_d7_retention_cohorts' THEN 4
                ELSE 5
            END,
            position
    ) TO '{validation_output_tmp}' (FORMAT CSV, HEADER)
    """
)

cohort_output_tmp.replace(cohort_output_path)
curve_output_tmp.replace(curve_output_path)
validation_output_tmp.replace(validation_output_path)

print('Outputs written.')


In [ ]:
sanity_row = sanity_checks.iloc[0].to_dict()
warnings = []

warning_map = {
    'negative_input_level_gain_day_count': 'negative_level_gain_day_present_in_input',
    'negative_cohort_age_rows': 'negative_cohort_age_rows_present',
    'curve_nonpositive_cohort_size_rows': 'curve_nonpositive_cohort_size_rows_present',
    'curve_retained_gt_cohort_size_rows': 'curve_retained_gt_cohort_size_rows_present',
    'curve_retention_rate_out_of_bounds_rows': 'curve_retention_rate_out_of_bounds_rows_present',
    'summary_nonpositive_cohort_size_rows': 'summary_nonpositive_cohort_size_rows_present',
    'summary_retention_rate_out_of_bounds_rows': 'summary_retention_rate_out_of_bounds_rows_present',
    'immature_d1_populated_rows': 'immature_d1_rows_were_calculated',
    'immature_d7_populated_rows': 'immature_d7_rows_were_calculated',
    'immature_d14_populated_rows': 'immature_d14_rows_were_calculated',
    'immature_d30_populated_rows': 'immature_d30_rows_were_calculated',
}

for key, warning_text in warning_map.items():
    if int(sanity_row.get(key, 0) or 0) > 0:
        warnings.append(warning_text)

warning_summary = 'none' if not warnings else '; '.join(warnings)
files_created = '; '.join([
    str(cohort_output_path),
    str(curve_output_path),
    str(validation_output_path),
])

final_report = con.execute(
    '''
    SELECT 'input_source_used' AS metric_name, ? AS metric_value
    UNION ALL
    SELECT 'total_unique_avatars', CAST((SELECT unique_avatars FROM retention_source_validation) AS VARCHAR)
    UNION ALL
    SELECT 'dataset_date_range', CAST((SELECT min_activity_date FROM retention_source_validation) AS VARCHAR) || ' -> ' || CAST((SELECT max_activity_date FROM retention_source_validation) AS VARCHAR)
    UNION ALL
    SELECT 'total_cohorts_before_filtering', CAST((SELECT total_cohorts_before_filtering FROM retention_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'total_cohorts_after_burn_in', CAST((SELECT total_cohorts_after_burn_in_filtering FROM retention_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'average_d1_retention', CAST((SELECT average_d1_retention FROM retention_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'average_d7_retention', CAST((SELECT average_d7_retention FROM retention_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'average_d14_retention', CAST((SELECT average_d14_retention FROM retention_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'average_d30_retention', CAST((SELECT average_d30_retention FROM retention_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'data_quality_warnings', ?
    UNION ALL
    SELECT 'files_created', ?
    '''
    , [str(fct_source_path), warning_summary, files_created]
).df()

final_report
